# Notebook 05_07: Quantization Deep Dive

**Learning Objectives:**
- Understand advanced quantization algorithms: GPTQ, AWQ, and bitsandbytes
- Compare INT8 vs INT4 quantization trade-offs on model quality and speed
- Load pre-quantized models from HuggingFace Hub
- Evaluate quantization quality using perplexity and task accuracy
- Choose the right quantization strategy for your deployment scenario

**Prerequisites:** Notebook 05_06 (Quantization & Compression intro)

## Prerequisites

### Hardware Requirements

| Requirement | CPU Path | GPU Path |
|-------------|----------|----------|
| **RAM** | 8 GB minimum | 8 GB |
| **VRAM** | N/A | 6 GB+ (RTX 4080 recommended) |
| **Storage** | 2 GB free | 4 GB free |
| **OS** | Any | Linux recommended for bitsandbytes |

### Software Requirements

```bash
# Core (already installed)
pip install transformers torch

# Advanced quantization (optional — notebook provides fallbacks)
pip install bitsandbytes          # 8-bit / 4-bit on CUDA
pip install auto-gptq             # GPTQ quantization
pip install autoawq               # AWQ quantization
pip install accelerate            # Required for device_map="auto"
```

> **Note:** `bitsandbytes`, `auto-gptq`, and `autoawq` require a CUDA GPU on Linux for full functionality. This notebook provides **conceptual walkthroughs and pre-quantized model loading** so CPU-only students can still learn the algorithms.

## Expected Behaviors

### With GPU (CUDA + Linux)
- bitsandbytes loads models in 4-bit/8-bit directly
- GPTQ and AWQ models load from Hub with quantized weights
- Perplexity evaluation runs in 30-60 seconds per model variant

### CPU-Only / Windows
- bitsandbytes may not be available (graceful skip)
- Pre-quantized GPTQ/AWQ models still load if `auto-gptq` / `autoawq` is installed
- Some sections show conceptual output instead of live results
- All algorithm explanations and comparison tables work regardless of hardware

### Common Issues
- **bitsandbytes import error on Windows**: Expected — library requires Linux + CUDA
- **CUDA out of memory**: Use the smaller model option or reduce batch size
- **Slow first load**: Quantized models download once, then use cache

## Overview

In Notebook 05_06, we covered **basic quantization**: FP16, PyTorch dynamic quantization, and ONNX export. These techniques are easy to apply but limited in compression ratio — typically 2-4x size reduction.

This notebook explores **advanced quantization methods** that achieve 4-8x compression with minimal quality loss:

| Method | Bits | Calibration | Speed | Quality | Best For |
|--------|------|-------------|-------|---------|----------|
| **bitsandbytes (NF4)** | 4-bit | None (on-the-fly) | Fast load | Good | Interactive experimentation |
| **bitsandbytes (INT8)** | 8-bit | None (on-the-fly) | Fast load | Very good | Balanced quality/size |
| **GPTQ** | 4-bit | Requires dataset | Fast inference | Good | Deployment |
| **AWQ** | 4-bit | Requires dataset | Fastest inference | Best | Production deployment |

### How They Work (Simplified)

**bitsandbytes**: Quantizes weights on-the-fly during model loading. No calibration needed. Uses NormalFloat4 (NF4) data type optimized for normally-distributed neural network weights.

**GPTQ** (GPT-Quantization): Post-training quantization that processes weights layer-by-layer using a calibration dataset. Minimizes the output error of each layer independently using an approximate second-order method.

**AWQ** (Activation-Aware Weight Quantization): Observes which weights matter most by looking at activation magnitudes. Protects salient weights (top ~1%) by scaling them before quantization, achieving better quality than naive round-to-nearest.

## Setup and Installation

In [ ]:
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# Check optional dependencies
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
except ImportError:
    BNB_AVAILABLE = False
    print("bitsandbytes not available. Install with: pip install bitsandbytes")
    print("  (Requires Linux + CUDA GPU for full functionality)")

try:
    from auto_gptq import AutoGPTQForCausalLM
    GPTQ_AVAILABLE = True
except ImportError:
    GPTQ_AVAILABLE = False
    print("auto-gptq not available. Install with: pip install auto-gptq")

try:
    from awq import AutoAWQForCausalLM
    AWQ_AVAILABLE = True
except ImportError:
    AWQ_AVAILABLE = False
    print("autoawq not available. Install with: pip install autoawq")

try:
    import accelerate
    ACCELERATE_AVAILABLE = True
except ImportError:
    ACCELERATE_AVAILABLE = False
    print("accelerate not available. Install with: pip install accelerate")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU memory: {gpu_mem:.1f} GB")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if BNB_AVAILABLE:
    print(f"bitsandbytes: {bnb.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

We use a small causal language model so quantization experiments run quickly. GPT-2 is ideal because it is small enough for CPU yet large enough that quantization makes a measurable difference.

In [ ]:
# Option 1: Small model (CPU-friendly, ~500 MB)
MODEL_NAME = "gpt2"

# Option 2: Medium model (GPU recommended, ~1.5 GB)
# MODEL_NAME = "gpt2-medium"

# Option 3: Large model (GPU required, ~3 GB)
# MODEL_NAME = "gpt2-large"

print(f"Selected model: {MODEL_NAME}")

## Part 1: Quantization Algorithms Explained

Before loading quantized models, let's understand *how* each algorithm works. The core challenge is the same: represent FP32 weights using fewer bits while minimizing the error introduced.

### Round-to-Nearest (RTN)

The simplest approach: map each FP32 weight to the nearest INT value in a reduced range. This is what PyTorch dynamic quantization does. It works well for 8-bit but degrades significantly at 4-bit because the quantization grid is too coarse.

### GPTQ: Layer-by-Layer Optimization

GPTQ improves on RTN by considering how quantization errors propagate through each layer. For each layer, it:
1. Feeds calibration data through the model to measure activations
2. Quantizes weights one column at a time
3. Compensates for each column's error by adjusting the remaining (not-yet-quantized) columns
4. Uses an approximate Hessian (second-order information) to decide the best compensation

This "error compensation" is why GPTQ achieves much better quality than RTN at 4-bit.

### AWQ: Protect the Important Weights

AWQ takes a different approach — instead of compensating for errors, it *prevents* errors on the weights that matter most:
1. Feeds calibration data and records activation magnitudes
2. Identifies the top ~1% of "salient" weight channels (those with large activations)
3. Scales these channels *up* before quantization (so rounding error is proportionally smaller)
4. Scales them back down during inference

This is elegant because it requires no changes to the quantization algorithm itself — just a pre-processing step.

### bitsandbytes NF4: Optimal Data Type

bitsandbytes NF4 (NormalFloat4) is designed for the observation that neural network weights follow a roughly normal distribution:
1. Instead of uniform quantization levels, NF4 places quantization bins according to the normal distribution
2. More bins near zero (where most weights are) and fewer bins in the tails
3. Combined with double quantization (quantizing the quantization constants too) for extra savings

In [ ]:
# Demonstrate the quantization error concept
print("=== QUANTIZATION ERROR DEMONSTRATION ===\n")

# Simulate weight quantization at different bit widths
np.random.seed(SEED)
weights = np.random.randn(1000).astype(np.float32)  # Normally distributed weights

def simulate_quantization(values: np.ndarray, num_bits: int) -> tuple[np.ndarray, float]:
    """Simulate symmetric quantization at a given bit width.

    Args:
        values: Array of FP32 values to quantize.
        num_bits: Number of bits for quantization.

    Returns:
        Tuple of (quantized values, mean squared error).
    """
    num_levels = 2 ** num_bits
    val_min, val_max = values.min(), values.max()
    scale = (val_max - val_min) / (num_levels - 1)
    quantized = np.round((values - val_min) / scale) * scale + val_min
    mse = np.mean((values - quantized) ** 2)
    return quantized, mse


results = []
for bits in [8, 4, 3, 2]:
    _, mse = simulate_quantization(weights, bits)
    results.append({"Bits": bits, "Levels": 2**bits, "MSE": f"{mse:.6f}", "Relative Error": f"{mse / np.var(weights):.4%}"})

error_df = pd.DataFrame(results)
print(error_df.to_string(index=False))
print("\nNotice: Error increases dramatically below 4 bits.")
print("This is why advanced algorithms (GPTQ, AWQ) are needed for 4-bit quantization.")

## Part 2: Baseline FP32 Model

Let's load the full-precision model and establish baseline metrics for size, speed, and text generation quality.

In [ ]:
print(f"Loading baseline model: {MODEL_NAME} (FP32)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_fp32.eval()


def get_model_size_mb(model: torch.nn.Module) -> float:
    """Calculate model size in megabytes.

    Args:
        model: PyTorch model.

    Returns:
        Size in MB.
    """
    param_bytes = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024 ** 2)


def count_parameters(model: torch.nn.Module) -> int:
    """Count total model parameters.

    Args:
        model: PyTorch model.

    Returns:
        Number of parameters.
    """
    return sum(p.numel() for p in model.parameters())


fp32_size = get_model_size_mb(model_fp32)
print(f"Parameters: {count_parameters(model_fp32):,}")
print(f"FP32 size:  {fp32_size:.1f} MB")

In [ ]:
def generate_text(
    model: torch.nn.Module,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 50,
) -> str:
    """Generate text from a prompt.

    Args:
        model: Causal language model.
        tokenizer: Tokenizer for the model.
        prompt: Input text prompt.
        max_new_tokens: Maximum tokens to generate.

    Returns:
        Generated text string.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Test baseline generation
test_prompts = [
    "The future of artificial intelligence is",
    "In a distant galaxy, scientists discovered",
    "The most important lesson in machine learning is",
]

print("=== BASELINE GENERATION (FP32) ===\n")
for prompt in test_prompts:
    output = generate_text(model_fp32, tokenizer, prompt)
    print(f"Prompt: {prompt}")
    print(f"Output: {output}\n")

## Part 3: PyTorch Dynamic Quantization (INT8 Baseline)

As a reference point for the advanced methods, let's apply PyTorch's built-in dynamic quantization. This converts Linear layer weights to INT8 at load time — no calibration data needed, works on any CPU.

In [ ]:
# Apply dynamic quantization (CPU only — this is our baseline advanced method)
model_int8_dynamic = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},
    dtype=torch.qint8,
)

int8_size = get_model_size_mb(model_int8_dynamic)
print("=== PYTORCH DYNAMIC QUANTIZATION (INT8) ===")
print(f"FP32 size:  {fp32_size:.1f} MB")
print(f"INT8 size:  {int8_size:.1f} MB")
print(f"Reduction:  {fp32_size / int8_size:.1f}x")

# Test generation
print("\n=== INT8 GENERATION ===\n")
for prompt in test_prompts:
    output = generate_text(model_int8_dynamic, tokenizer, prompt)
    print(f"Prompt: {prompt}")
    print(f"Output: {output}\n")

## Part 4: bitsandbytes Quantization (4-bit / 8-bit)

bitsandbytes is the most popular quantization library in the HuggingFace ecosystem. It quantizes models *during loading* — no separate calibration step needed. This makes it extremely convenient for experimentation.

Two key modes:
- **8-bit (LLM.int8())**: Keeps outlier features in FP16, quantizes the rest to INT8. Near-lossless.
- **4-bit (NF4)**: Uses NormalFloat4 data type optimized for normally-distributed weights. Combined with double quantization for extra compression.

> **Hardware requirement:** bitsandbytes requires a CUDA GPU on Linux. If you are on CPU or Windows, this section will show the configuration and expected output.

In [ ]:
CUDA_AVAILABLE = torch.cuda.is_available() and BNB_AVAILABLE and ACCELERATE_AVAILABLE

if CUDA_AVAILABLE:
    # ── 8-bit quantization ──
    print("Loading model in 8-bit (LLM.int8())...")
    quantization_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
    model_8bit = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config_8bit,
        device_map="auto",
    )
    size_8bit = get_model_size_mb(model_8bit)
    print(f"8-bit size: {size_8bit:.1f} MB  (reduction: {fp32_size / size_8bit:.1f}x)")

    print("\n=== 8-BIT GENERATION ===\n")
    for prompt in test_prompts:
        output = generate_text(model_8bit, tokenizer, prompt)
        print(f"Prompt: {prompt}")
        print(f"Output: {output}\n")

    del model_8bit
    torch.cuda.empty_cache()
else:
    print("bitsandbytes 8-bit requires CUDA GPU + Linux. Showing configuration:\n")
    print('quantization_config_8bit = BitsAndBytesConfig(load_in_8bit=True)')
    print('model_8bit = AutoModelForCausalLM.from_pretrained(')
    print('    MODEL_NAME,')
    print('    quantization_config=quantization_config_8bit,')
    print('    device_map="auto",')
    print(')')
    print("\nExpected: ~2x size reduction with near-lossless quality")

In [ ]:
if CUDA_AVAILABLE:
    # ── 4-bit quantization (NF4 with double quantization) ──
    print("Loading model in 4-bit (NF4 + double quantization)...")
    quantization_config_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4 data type
        bnb_4bit_use_double_quant=True,       # Quantize the quantization constants
        bnb_4bit_compute_dtype=torch.float16, # Compute in FP16 for speed
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config_4bit,
        device_map="auto",
    )
    size_4bit = get_model_size_mb(model_4bit)
    print(f"4-bit size: {size_4bit:.1f} MB  (reduction: {fp32_size / size_4bit:.1f}x)")

    print("\n=== 4-BIT NF4 GENERATION ===\n")
    for prompt in test_prompts:
        output = generate_text(model_4bit, tokenizer, prompt)
        print(f"Prompt: {prompt}")
        print(f"Output: {output}\n")

    del model_4bit
    torch.cuda.empty_cache()
else:
    print("bitsandbytes 4-bit requires CUDA GPU + Linux. Showing configuration:\n")
    print('quantization_config_4bit = BitsAndBytesConfig(')
    print('    load_in_4bit=True,')
    print('    bnb_4bit_quant_type="nf4",           # NormalFloat4 data type')
    print('    bnb_4bit_use_double_quant=True,       # Quantize the quantization constants')
    print('    bnb_4bit_compute_dtype=torch.float16, # Compute in FP16 for speed')
    print(')')
    print("\nExpected: ~4-8x size reduction with minor quality loss")

## Part 5: Loading Pre-Quantized Models from Hub

The HuggingFace Hub hosts thousands of pre-quantized models — you can use GPTQ or AWQ models without running the quantization yourself. This is the most practical approach for most users: someone else did the expensive calibration step, and you just load the result.

### Finding Quantized Models

Search the Hub for quantized variants:
- Search for `model-name GPTQ` or `model-name AWQ`
- Filter by the `gptq` or `awq` tags
- Popular quantizers: TheBloke, Hugging Face Optimum

### How It Works

Pre-quantized models store INT4 weights directly in the checkpoint. When you load them with the right library installed (`auto-gptq` or `autoawq`), the weights are unpacked and used for efficient inference. No calibration needed on your end.

In [ ]:
# Demonstrate loading a pre-quantized GPTQ model
# We use a small model to keep downloads manageable

GPTQ_MODEL = "TheBloke/gpt2-GPTQ"  # Pre-quantized GPT-2

if GPTQ_AVAILABLE and torch.cuda.is_available():
    print(f"Loading pre-quantized GPTQ model: {GPTQ_MODEL}...")
    try:
        model_gptq = AutoModelForCausalLM.from_pretrained(
            GPTQ_MODEL,
            device_map="auto",
        )
        size_gptq = get_model_size_mb(model_gptq)
        print(f"GPTQ size: {size_gptq:.1f} MB  (reduction: {fp32_size / max(size_gptq, 0.1):.1f}x)")

        print("\n=== GPTQ GENERATION ===\n")
        for prompt in test_prompts:
            output = generate_text(model_gptq, tokenizer, prompt)
            print(f"Prompt: {prompt}")
            print(f"Output: {output}\n")

        del model_gptq
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Could not load GPTQ model: {e}")
        print("This is expected if the model is not available or GPU is insufficient.")
else:
    print(f"GPTQ loading requires auto-gptq + CUDA GPU.")
    print(f"\nTo load a pre-quantized model:")
    print(f'  model = AutoModelForCausalLM.from_pretrained("{GPTQ_MODEL}", device_map="auto")')
    print(f"\nThe transformers library auto-detects the GPTQ config in the model repository")
    print(f"and uses auto-gptq for efficient inference.")

## Part 6: Evaluating Quantization Quality

How do we measure whether quantization hurts model quality? The standard metric for language models is **perplexity** — a measure of how surprised the model is by a text. Lower perplexity means the model assigns higher probability to the correct next tokens.

$$\text{Perplexity} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i | w_1, \ldots, w_{i-1})\right)$$

A small increase in perplexity (e.g., from 20 to 21) is usually acceptable. A large increase (e.g., from 20 to 35) indicates significant quality loss.

In [ ]:
def compute_perplexity(
    model: torch.nn.Module,
    tokenizer: AutoTokenizer,
    texts: list[str],
    max_length: int = 128,
) -> float:
    """Compute perplexity of a model on a set of texts.

    Args:
        model: Causal language model.
        tokenizer: Tokenizer for the model.
        texts: List of text strings to evaluate.
        max_length: Maximum sequence length for tokenization.

    Returns:
        Average perplexity across all texts.
    """
    total_loss = 0.0
    total_tokens = 0

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss.item()

        num_tokens = inputs["input_ids"].shape[1]
        total_loss += loss * num_tokens
        total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    return float(np.exp(avg_loss))


# Evaluation texts (diverse topics for robust measurement)
eval_texts = [
    "The transformer architecture revolutionized natural language processing by introducing self-attention mechanisms that allow models to process entire sequences in parallel.",
    "Climate change poses significant challenges to global food security. Rising temperatures and shifting weather patterns affect crop yields and water availability worldwide.",
    "Python is a versatile programming language widely used in data science, web development, and artificial intelligence applications.",
    "The human brain contains approximately 86 billion neurons connected by trillions of synapses, forming the most complex known structure in the universe.",
    "Economic theories suggest that market equilibrium is reached when supply equals demand, though real-world markets rarely achieve perfect equilibrium.",
    "Quantum computing leverages quantum mechanical phenomena like superposition and entanglement to perform computations that would be infeasible for classical computers.",
    "The Renaissance period marked a cultural rebirth in Europe, with significant advances in art, architecture, science, and literature from the 14th to 17th centuries.",
    "Machine learning models require careful evaluation using appropriate metrics, cross-validation, and holdout test sets to ensure they generalize well to unseen data.",
]

print(f"Evaluating perplexity on {len(eval_texts)} texts...\n")

# FP32 baseline
ppl_fp32 = compute_perplexity(model_fp32, tokenizer, eval_texts)
print(f"FP32 perplexity:         {ppl_fp32:.2f}")

# INT8 dynamic
ppl_int8 = compute_perplexity(model_int8_dynamic, tokenizer, eval_texts)
print(f"INT8 dynamic perplexity: {ppl_int8:.2f}")
print(f"Degradation:             {((ppl_int8 - ppl_fp32) / ppl_fp32) * 100:+.2f}%")

## Part 7: Speed Benchmarking

Size reduction is important, but inference speed is what matters for deployment. Let's measure generation speed across quantization methods with proper warmup and multiple runs.

In [ ]:
def benchmark_generation(
    model: torch.nn.Module,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    num_warmup: int = 3,
    num_runs: int = 10,
) -> dict[str, float]:
    """Benchmark text generation speed.

    Args:
        model: Causal language model.
        tokenizer: Tokenizer for the model.
        prompt: Input prompt for generation.
        max_new_tokens: Tokens to generate per run.
        num_warmup: Warmup iterations (not timed).
        num_runs: Timed iterations.

    Returns:
        Dictionary with mean_ms, tokens_per_sec, and std_ms.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Warmup
    for _ in range(num_warmup):
        with torch.no_grad():
            _ = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )

    # Timed runs
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    mean_time = np.mean(times)

    return {
        "mean_ms": mean_time * 1000,
        "std_ms": np.std(times) * 1000,
        "tokens_per_sec": generated_tokens / mean_time,
    }


bench_prompt = "The future of artificial intelligence is"

print("Benchmarking FP32...")
bench_fp32 = benchmark_generation(model_fp32, tokenizer, bench_prompt)

print("Benchmarking INT8 dynamic...")
bench_int8 = benchmark_generation(model_int8_dynamic, tokenizer, bench_prompt)

print("\n=== GENERATION SPEED COMPARISON ===\n")
bench_df = pd.DataFrame([
    {
        "Model": "FP32 (baseline)",
        "Size (MB)": f"{fp32_size:.1f}",
        "Mean (ms)": f"{bench_fp32['mean_ms']:.1f}",
        "Std (ms)": f"{bench_fp32['std_ms']:.1f}",
        "Tokens/sec": f"{bench_fp32['tokens_per_sec']:.1f}",
        "Speedup": "1.0x",
    },
    {
        "Model": "INT8 dynamic",
        "Size (MB)": f"{int8_size:.1f}",
        "Mean (ms)": f"{bench_int8['mean_ms']:.1f}",
        "Std (ms)": f"{bench_int8['std_ms']:.1f}",
        "Tokens/sec": f"{bench_int8['tokens_per_sec']:.1f}",
        "Speedup": f"{bench_fp32['mean_ms'] / bench_int8['mean_ms']:.2f}x",
    },
])
print(bench_df.to_string(index=False))

## Part 8: Comprehensive Comparison

Let's compile all our measurements into a single decision-ready comparison table. This includes results from both CPU-available methods and expected results for GPU-only methods based on published benchmarks.

In [ ]:
# Build comprehensive comparison table
comparison_rows = [
    {
        "Method": "FP32 (baseline)",
        "Bits": 32,
        "Size (MB)": f"{fp32_size:.1f}",
        "Size Reduction": "1.0x",
        "Perplexity": f"{ppl_fp32:.2f}",
        "PPL Degradation": "baseline",
        "Calibration": "None",
        "Hardware": "Any",
    },
    {
        "Method": "PyTorch INT8",
        "Bits": 8,
        "Size (MB)": f"{int8_size:.1f}",
        "Size Reduction": f"{fp32_size / int8_size:.1f}x",
        "Perplexity": f"{ppl_int8:.2f}",
        "PPL Degradation": f"{((ppl_int8 - ppl_fp32) / ppl_fp32) * 100:+.1f}%",
        "Calibration": "None",
        "Hardware": "CPU",
    },
    {
        "Method": "bitsandbytes INT8",
        "Bits": 8,
        "Size (MB)": f"~{fp32_size / 2:.0f}",
        "Size Reduction": "~2x",
        "Perplexity": "~baseline",
        "PPL Degradation": "<0.5%",
        "Calibration": "None",
        "Hardware": "CUDA GPU",
    },
    {
        "Method": "bitsandbytes NF4",
        "Bits": 4,
        "Size (MB)": f"~{fp32_size / 4:.0f}",
        "Size Reduction": "~4x",
        "Perplexity": "~+2-5%",
        "PPL Degradation": "~2-5%",
        "Calibration": "None",
        "Hardware": "CUDA GPU",
    },
    {
        "Method": "GPTQ (4-bit)",
        "Bits": 4,
        "Size (MB)": f"~{fp32_size / 4:.0f}",
        "Size Reduction": "~4x",
        "Perplexity": "~+1-3%",
        "PPL Degradation": "~1-3%",
        "Calibration": "Dataset",
        "Hardware": "CUDA GPU",
    },
    {
        "Method": "AWQ (4-bit)",
        "Bits": 4,
        "Size (MB)": f"~{fp32_size / 4:.0f}",
        "Size Reduction": "~4x",
        "Perplexity": "~+0.5-2%",
        "PPL Degradation": "~0.5-2%",
        "Calibration": "Dataset",
        "Hardware": "CUDA GPU",
    },
]

print("=== COMPREHENSIVE QUANTIZATION COMPARISON ===\n")
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

print("\nNote: bitsandbytes/GPTQ/AWQ values marked with ~ are expected results from")
print("published benchmarks. Run on a CUDA GPU for live measurements.")

## Part 9: Choosing the Right Method

The right quantization method depends on your constraints. Here is a practical decision framework.

### Decision Tree

1. **No GPU?** → PyTorch dynamic quantization (INT8) or ONNX quantization
2. **GPU available, experimenting?** → bitsandbytes NF4 (easiest setup)
3. **GPU available, deploying?** → AWQ (best quality/speed ratio)
4. **Need maximum compatibility?** → GPTQ (widest library support)
5. **Need near-lossless?** → bitsandbytes INT8 or FP16

### Key Trade-offs

In [ ]:
tradeoffs = pd.DataFrame({
    "Priority": [
        "Easiest setup",
        "Best quality at 4-bit",
        "Fastest inference",
        "No calibration needed",
        "Works on CPU",
        "Widest model availability",
        "Best for fine-tuning (QLoRA)",
    ],
    "Recommended Method": [
        "bitsandbytes NF4",
        "AWQ",
        "AWQ",
        "bitsandbytes",
        "PyTorch dynamic (INT8)",
        "GPTQ (most Hub models)",
        "bitsandbytes NF4",
    ],
    "Why": [
        "One-line config, no calibration",
        "Activation-aware preserves quality",
        "Optimized kernels for 4-bit matmul",
        "Quantizes on-the-fly during load",
        "No CUDA required",
        "TheBloke + community uploads",
        "Designed for QLoRA training",
    ],
})

print("=== DECISION GUIDE ===\n")
print(tradeoffs.to_string(index=False))

## Cleanup

Let's free the models from memory.

In [ ]:
del model_fp32, model_int8_dynamic
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Models freed from memory.")

## Exercises

1. **Compare Perplexity on Domain Text**: Create evaluation texts from a specific domain (e.g., medical, legal) and check if quantization degrades more on specialized text than general text.

2. **Quantization Bit-Width Sweep**: Modify the `simulate_quantization` function to also test 5-bit, 6-bit, and 7-bit. Plot MSE vs bits to find the "sweet spot" where error drops sharply.

3. **bitsandbytes NF4 vs FP4**: If you have a CUDA GPU, compare `bnb_4bit_quant_type="nf4"` with `bnb_4bit_quant_type="fp4"`. Which produces lower perplexity?

4. **Load a Larger Quantized Model**: Find a GPTQ-quantized Llama or Mistral model on the Hub and load it. How does generation quality compare to GPT-2?

5. **Double Quantization Impact**: Compare bitsandbytes 4-bit with and without `bnb_4bit_use_double_quant=True`. How much extra compression does double quantization provide?

## Key Takeaways

- **PyTorch dynamic quantization** is the easiest method (INT8, CPU, no calibration) with ~2x size reduction
- **bitsandbytes** enables 4-bit/8-bit loading with a single config — ideal for experimentation and QLoRA training
- **GPTQ** uses calibration data and layer-by-layer optimization for high-quality 4-bit — widest Hub availability
- **AWQ** protects salient weights via activation-aware scaling — best quality/speed ratio for deployment
- **Perplexity** is the standard metric for evaluating quantization quality — measure before deploying

## Next Steps

- Try **Notebook 05_08**: Training Best Practices (Trainer API, scheduling, mixed precision)
- Explore [HuggingFace Quantization Docs](https://huggingface.co/docs/transformers/quantization)
- Browse pre-quantized models on [HuggingFace Hub](https://huggingface.co/models?other=gptq)

## Resources

- [bitsandbytes Documentation](https://huggingface.co/docs/bitsandbytes/)
- [GPTQ Paper](https://arxiv.org/abs/2210.17323)
- [AWQ Paper](https://arxiv.org/abs/2306.00978)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [HuggingFace Quantization Guide](https://huggingface.co/docs/transformers/quantization)